In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

### Newton's method for finding extrema

##### Single-variable case:

$$ f(x) \approx f(x_0) + f'(x_0)(x - x_0) + \frac{1}{2} f''(x_0)(x - x_0)^2 $$
$$ \frac{df}{dx} = 0 $$
$$ \frac{df}{dx} = f'(x_0) + f''(x_0)(x - x_0) $$
$$ x = x_0 - \frac{f'(x_0)}{f''(x_0)} $$
$$ \\[2em] $$

##### Example 1:

$$ f(x) = x^2 $$
$$ f'(x) = 2x $$
$$ f''(x) = 2 $$
$$ x_{\text{new}} = x - \frac{2x}{2} = 0 $$
$$ \\[2em] $$

##### Example 2:

$$ f(x) = 2x^2 + 3x - 1 $$
$$ f'(x) = 4x + 3 $$
$$ f''(x) = 4 $$
$$ x_{\text{new}} = x - \frac{4x + 3}{4} = -\frac{3}{4} $$
$$ \\[2em] $$

In [ ]:
from math import sin

import torch as tr
import torch.autograd as autograd

import import_ipynb
from common import check_eq, assert_eq # type: ignore


def newton_1d(f: callable, x0: float, epochs=20, atol=0.01) -> float:
    """
    Perform Newton's method to find a local minimum of a single-variable function `f` starting from an initial guess `x0`.
    
    Args:
        f: A single-variable function that takes a Tensor input and returns a scalar Tensor output.
        x0: A float representing the initial guess for the minimum.
        epochs: The maximum number of iterations to perform.
        atol: The absolute tolerance for convergence. The method will stop if the norm of the change in `x` is less than `atol`.
    
    Returns:
        A float representing the estimated location of the minimum of `f`.
    """
    
    x = tr.tensor(x0, dtype=tr.float32, requires_grad=True)
    check_eq(x.shape, [])

    for _ in range(epochs):
        y = f(x)
        assert_eq(y.shape, ())

        (dfdx, ) = autograd.grad(y, x, create_graph=True)
        check_eq(dfdx.shape, [])

        (dfdx2, ) = autograd.grad(dfdx, x, create_graph=False)
        check_eq(dfdx.shape, [])

        if check_eq(dfdx, 0.0, atol=1e-6):
            break

        x_new = x - dfdx / dfdx2
        check_eq(x_new.shape, [])

        if check_eq(x, x_new, atol=atol):
            return x_new.item()

        x = x_new.detach().requires_grad_()

    return x.item()


def pow2(x: tr.Tensor) -> tr.Tensor:
    return x ** 2


def test_newton_1d() -> None:
    # it works with tensor functions
    assert_eq(newton_1d(pow2, x0=1.0), 0.0, atol=0.01)

    # it works with lambdas due to `lambda` → `tensor function` conversion on the fly
    assert_eq(newton_1d(lambda x: 2*x**2 + 3*x - 1, x0=1.0), -0.75, atol=0.01)

    # it does not work with non-tensor functions, 
    # because they do not have a graph to compute the derivatives
    # assert_eq(newton_1d(lambda x: sin(x), x0=1.0), -0.75, atol=0.01)


if __name__ == "__main__":
    test_newton_1d()

### Newton's method for finding extrema

##### Multivariate case:

Instead of a derivative, we have a gradient

$$ \nabla f(\mathbf{x}) = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix} $$

and instead of a second derivative, we have a Hessian matrix

$$ H(\mathbf{x}) = \begin{bmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} & \cdots & \frac{\partial^2 f}{\partial x_1 \partial x_n} \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} & \cdots & \frac{\partial^2 f}{\partial x_2 \partial x_n} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial^2 f}{\partial x_n \partial x_1} & \frac{\partial^2 f}{\partial x_n \partial x_2} & \cdots & \frac{\partial^2 f}{\partial x_n^2} \end{bmatrix} $$

The Taylor expansion in multiple dimensions is:

$$ f(\mathbf{x}) \approx f(\mathbf{x}_0) + \nabla f(\mathbf{x}_0)^T (\mathbf{x} - \mathbf{x}_0) + \frac{1}{2} (\mathbf{x} - \mathbf{x}_0)^T H(\mathbf{x}_0) (\mathbf{x} - \mathbf{x}_0) $$
$$ \nabla f(\mathbf{x}) = 0 $$
$$ \mathbf{x} = \mathbf{x}_0 - H(\mathbf{x}_0)^{-1} \nabla f(\mathbf{x}_0) $$
$$ \\[2em] $$

##### Example 1:

$$ f(x, y) = x^2 + y^2 $$
$$ \nabla f(x, y) = \begin{bmatrix} 2x \\ 2y \end{bmatrix} $$
$$ H(x, y) = \begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix} $$
$$ H(x, y)^{-1} = \begin{bmatrix} \frac{1}{2} & 0 \\ 0 & \frac{1}{2} \end{bmatrix} $$
$$ \mathbf{x}_{\text{new}} = \begin{bmatrix} x \\ y \end{bmatrix} - \begin{bmatrix} \frac{1}{2} & 0 \\ 0 & \frac{1}{2} \end{bmatrix} \begin{bmatrix} 2x \\ 2y \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix} $$
$$ \\[2em] $$

##### Example 2:

$$ f(x, y) = x^2 + y^2 + 3x + 4y - 1 $$
$$ \nabla f(x, y) = \begin{bmatrix} 2x + 3 \\ 2y + 4 \end{bmatrix} $$
$$ H(x, y) = \begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix} $$
$$ H(x, y)^{-1} = \begin{bmatrix} \frac{1}{2} & 0 \\ 0 & \frac{1}{2} \end{bmatrix} $$
$$ \mathbf{x}_{\text{new}} = \begin{bmatrix} x \\ y \end{bmatrix} - \begin{bmatrix} \frac{1}{2} & 0 \\ 0 & \frac{1}{2} \end{bmatrix} \begin{bmatrix} 2x + 3 \\ 2y + 4 \end{bmatrix} = \begin{bmatrix} -\frac{3}{2} \\ -2 \end{bmatrix} $$

In [ ]:
from math import sin

import torch as tr
import torch.linalg as linalg
import torch.autograd as autograd
import torch.autograd.functional as fun

import import_ipynb
from common import assert_eq, assert_eq, T # type: ignore


def newton_nd(f: callable, x0: tr.Tensor, epochs=20, atol=1e-6) -> tr.Tensor:
    """
    Perform Newton's method to find a local minimum of a multivariate function `f` starting from an initial guess `x0`.
    
    Args:
        f: A multivariate function that takes a Tensor input and returns a scalar Tensor output.
        x0: A Tensor representing the initial guess for the minimum. It should be a 1D Tensor of shape (n,).
        epochs: The maximum number of iterations to perform.
        atol: The absolute tolerance for convergence. The method will stop if the norm of the change in `x` is less than `atol`.
    
    Returns:
        A Tensor representing the estimated location of the minimum of `f`.
    """
    
    x = x0.clone().detach().requires_grad_()
    n = x.shape[0]
    assert_eq(x.shape, (n,))

    for _ in range(epochs):
        y = f(x)
        assert_eq(y.shape, ())
        
        grad_x = autograd.grad(y, x, create_graph=True)[0]
        assert_eq(grad_x.shape,(n,))

        hessian_x = fun.hessian(f, x)
        assert_eq(hessian_x.shape, (n, n))

        delta = linalg.solve(hessian_x, grad_x)
        assert_eq(delta.shape, (n,))

        x_new = x - delta
        assert_eq(x_new.shape, (n,))

        if tr.norm(x_new - x) < atol:
            return x_new.detach()

        x = x_new.detach().requires_grad_()

    return x.detach()



def pow2(vector: tr.Tensor) -> tr.Tensor:
    return (vector ** 2).sum()


def test_newton_nd() -> None:
    # it works with tensor functions
    assert_eq(newton_nd(pow2, x0=T([1.0, 1.0])), T([0.0, 0.0]), atol=0.01)

    # it works with lambdas due to `lambda` → `tensor function` conversion on the fly
    assert_eq(newton_nd(lambda xy: xy[0]**2 + xy[1]**2 + 3*xy[0] + 4*xy[1] - 1, x0=T([1.0, 1.0])), T([-1.5, -2.0]), atol=0.01)

    # it does not work with non-tensor functions, 
    # because they do not have a graph to compute the derivatives
    # assert_eq(newton_nd(library.pow2, x0=T([1.0, 1.0])), T([0.0, 0.0]), atol=0.01)

if __name__ == "__main__":
    test_newton_nd()